# LAB 08 Guide — Lakeflow Jobs: Triggers, Dependencies & Orchestration

## Scenario

> *"Orchestrate your Medallion pipeline by creating multi-task Databricks Jobs with dependencies, triggers, and monitoring. First build a pipeline in the UI (Workshop), then evolve the job and verify every change with the Databricks SDK (Engineering)."*



## Objectives

After completing this lab you will be able to:
- Create and configure multi-task Jobs in Databricks UI
- Set task dependencies (DAG structure)
- Configure Table Update Triggers for event-driven orchestration
- Add `for_each` and If/else condition tasks (control flow)
- Read job settings, triggers and run history with the Databricks SDK
- Use Repair Runs for partial re-execution
- Write Quartz cron expressions for scheduling
- Choose between Job Clusters and All-Purpose Clusters



## Prerequisites

- Serverless compute attached to the notebook
- lab_07 done
- Medallion notebooks (`materials/medallion/`) available in your Git folder



## Section 1 — Workshop: Creating & Running Jobs in the Databricks UI

Follow the trainer's instructions step by step.

### Step 1: Create a New Job
- Go to **Jobs & Pipelines → Create → Job**

### Step 2: Set the Job Name
- Name it `<your_name>_customer_pipeline`

### Step 3: Configure Job Parameters
- `catalog` = `retailhub_<your_name>`
- `source_path` = `/Volumes/retailhub_<your_name>/default/datasets`

### Step 4: Add First Task — `bronze_customer`
- Type: **Notebook** task
- Path: `<your Git folder>/materials/medallion/bronze_customers`
- No dependencies (first task)

### Step 5: Add Second Task — `silver_customer`
- Type: **Notebook** task
- Path: `<your Git folder>/materials/medallion/silver_customers`
- **Depends on:** `bronze_customer`

### Step 6: Create Second Job — `orders_pipeline`
- Same job parameters as Step 3; 4 Notebook tasks with dependencies:

| Task key | Notebook path | Depends on |
|----------|---------------|------------|
| `bronze_orders` | `<your Git folder>/materials/medallion/bronze_orders` | — |
| `silver_orders` | `<your Git folder>/materials/medallion/silver_orders_cleaned` | `bronze_orders` |
| `gold_daily_orders` | `<your Git folder>/materials/medallion/gold_daily_orders` | `silver_orders` |
| `gold_summary` | `<your Git folder>/materials/medallion/gold_customer_orders_summary` | `gold_daily_orders` |

### Step 7: Configure Table Update Trigger
- Job page → **Schedules & Triggers** → **Table update** trigger on `retailhub_<your_name>.silver.silver_customers`

### Step 8: Run the Pipeline
1. Run `customer_pipeline` first
2. `orders_pipeline` should trigger automatically via Table Update Trigger
3. Monitor execution in the **Runs** tab



## Section 2 — Engineering: Evolve the Job, Verify with the SDK

Open the lab notebook **`lab_08_orchestration.ipynb`** (in `notebooks/day3/lab/`) and complete the `# TODO` cells.

| Task | What to do | Key concept |
|------|-----------|-------------|
| **Task A** | Add a `for_each` task over a list parameter | `for_each_task.inputs`, `{{input}}` |
| **Task B** | Add an If/else condition task | `condition_task` (left, op, right) + `depends_on` outcome |
| **Task C** | Point the table-update trigger at `bronze_customers` | raw Jobs API JSON: `trigger["table_update"]["table_names"]` |
| **Task D** | Break a task, then Repair run | `repair_history` has > 1 entry |
| **Task E** | Success rate from run history | `system.lakeflow.job_run_timeline` (fallback: `w.jobs.list_runs`) |
| **Closing check** | Quartz cron + compute selection | `0 0 8 ? * MON-FRI`, job cluster vs all-purpose |


## Detailed Hints

### Task A — `for_each`
- `orders_job = get_job(ORDERS_JOB)`
- `foreach_tasks = [t for t in orders_job.settings.tasks if t.for_each_task is not None]`
- List elements must be fully qualified table names — each iteration passes `{{input}}` to `spark.table(...)`

### Task B — If/else condition
- Add job parameter `min_rows_ok` = `true` first; `send_report` path: `<your Git folder>/materials/orchestration/task_03_report`
- Same comprehension with `t.condition_task is not None`
- API operators: `EQUAL_TO`, `NOT_EQUAL`, `GREATER_THAN`, `GREATER_THAN_OR_EQUAL`, `LESS_THAN`, `LESS_THAN_OR_EQUAL`
- Downstream tasks pick a branch with `depends_on: [{task_key: check_row_count, outcome: "true"}]`

### Task C — Trigger
- `trigger = raw_settings.get("trigger")` *(a dict, or `None` when no trigger is configured)*
- `table_names = raw_settings["trigger"]["table_update"]["table_names"]` *(raw JSON via `w.api_client.do(...)` — the serverless SDK may not map this field)*

### Task D — Repair run
- Break `gold_summary` via its **notebook path** (not a parameter — job parameters override task parameters)
- Loop `w.jobs.list_runs(job_id=orders_job.job_id, limit=25)` → `w.jobs.get_run(r.run_id, include_history=True)`
- Keep runs where `len(full.repair_history) > 1`

### Task E — Run history
- One row per run: `GROUP BY run_id` with `MAX_BY(result_state, period_end_time) AS result_state` (final state) from `system.lakeflow.job_run_timeline`, filtered `WHERE workspace_id = '{ws_id}' AND job_id = '{orders_job.job_id}'`; keep rows where `result_state IS NOT NULL`
- Why: runs longer than 1 h span several timeline rows, and a repaired run has rows for the failed attempt and the repair — it must count once, with its final state
- Fallback: `[r.state.result_state.value for r in w.jobs.list_runs(job_id=..., limit=50) if r.state and r.state.result_state]`

### Task values (reference, used by `materials/orchestration/`)
- Upstream: `dbutils.jobs.taskValues.set(key="row_count", value=n)`
- Downstream: `dbutils.jobs.taskValues.get(taskKey="validate", key="row_count", default=0, debugValue=0)`
- In task settings: `{{tasks.validate.values.row_count}}`

### Closing check: Quartz cron
- Lakeflow Jobs use **Quartz** cron: `seconds minutes hours day-of-month month day-of-week [year]`
- `0 0 6 * * ?` = daily at 6:00 · `0 0 * * * ?` = every hour · `0 0/15 * * * ?` = every 15 minutes
- Weekdays: day-of-week `MON-FRI` and day-of-month `?`
- **Job cluster**: scheduled production ETL · **All-purpose cluster**: interactive development


## Summary

In this lab you:
- Built multi-task Jobs with dependencies in the Databricks UI
- Configured Table Update Triggers for event-driven orchestration
- Added `for_each` and If/else condition tasks and verified them with the SDK
- Used Repair Runs for cost-efficient re-execution
- Computed a success rate from run history (system tables / Jobs API)
- Practiced Quartz cron expressions and compute selection

> **Exam Tip:** Repair Runs only re-execute the failed task and its downstream dependencies — upstream tasks are skipped. Job Clusters are cheaper for production. `system.lakeflow.job_run_timeline` is the key system table for job monitoring. Job schedules use Quartz cron (`0 0 6 * * ?`).

> **What's next:** In LAB 09 you will deploy the RetailHub pipeline and job as a Declarative Automation Bundle.
